# Pj1 · Deteksi bola, ring, dan papan basket dengan YOLO (MicroDuck)
RET503 Computer Vision and Deep Learning · Zahra Nabila Putri (4222401028)

**Sebelum mulai**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `dataset_v1.zip` ke Google Drive di folder `MyDrive/pj1/`.
3. Jalankan sel dari atas ke bawah. Eksperimen E1, E2, E3 disimpan ke Drive, jadi aman kalau sesi Colab terputus (jalankan ulang sel setup lalu lanjut ke eksperimen yang belum selesai).

## 1. Setup

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/pj1/dataset_v1.zip'
!rm -rf /content/dataset_v1
!unzip -q "{ZIP}" -d /content/
!ls /content/dataset_v1

## 2. Periksa dataset (jumlah gambar dan kotak per kelas)

In [ ]:
import glob, collections
D = '/content/dataset_v1/data.yaml'
NAMA = ['backboard', 'ball', 'rim']
for s in ['train', 'val', 'test']:
    imgs = glob.glob(f'/content/dataset_v1/images/{s}/*.jpg')
    c = collections.Counter()
    for f in glob.glob(f'/content/dataset_v1/labels/{s}/*.txt'):
        for l in open(f):
            if l.strip():
                c[NAMA[int(l.split()[0])]] += 1
    print(f'{s:5s} {len(imgs):4d} gambar | kotak:', dict(c))
print(open(D).read())

## 3. Pengaturan (ubah di sini saja)

In [ ]:
NIM, NAMA_MHS = '4222401028', 'Zahra Nabila Putri'
RUNS = '/content/drive/MyDrive/pj1/runs'      # hasil disimpan di Drive
EPOCHS, BATCH, SEED = 30, 16, 0
IMG_BASE = 416      # E1 dan E2
IMG_TUNE = 640      # E3: SATU perubahan = ukuran input (ganti ke 320 jika ingin uji versi ringan)

from ultralytics import YOLO
MODEL = 'yolo26n'   # sesuai slide; otomatis pindah ke yolo11n jika tidak tersedia
try:
    YOLO(f'{MODEL}.pt')
except Exception as e:
    print('yolo26n tidak tersedia, pakai yolo11n:', str(e)[:80])
    MODEL = 'yolo11n'
print('Model dipakai:', MODEL)

common = dict(data=D, epochs=EPOCHS, batch=BATCH, seed=SEED,
              project=RUNS, exist_ok=True, workers=2, plots=True)

## 4. Eksperimen
Data, epoch, batch, dan seed sama. Yang berbeda hanya satu faktor per eksperimen.

In [ ]:
# E1: baseline, pretrained COCO (transfer learning)
YOLO(f'{MODEL}.pt').train(imgsz=IMG_BASE, name='E1_pretrained', **common)

In [ ]:
# E2: arsitektur sama, bobot acak (tanpa transfer learning)
YOLO(f'{MODEL}.yaml').train(imgsz=IMG_BASE, pretrained=False, name='E2_scratch', **common)

In [ ]:
# E3: pretrained + SATU perubahan (imgsz)
YOLO(f'{MODEL}.pt').train(imgsz=IMG_TUNE, name='E3_tuning', **common)

## 5. Log eksperimen (data VAL)
Kolom `ms_GPU` hanya pembanding antar-eksperimen di GPU Colab. **FPS dan latensi untuk laporan harus diukur di Orange Pi 3B.**

In [ ]:
import pandas as pd
EKS = [('E1_pretrained', '-', IMG_BASE),
       ('E2_scratch', 'bobot acak', IMG_BASE),
       ('E3_tuning', f'imgsz={IMG_TUNE}', IMG_TUNE)]
rows, perkelas = [], {}
for nm, ubah, im in EKS:
    r = YOLO(f'{RUNS}/{nm}/weights/best.pt').val(data=D, split='val', imgsz=im, plots=False, verbose=False)
    rows.append(dict(eksperimen=nm, perubahan=ubah, dataset='v1', mAP50=r.box.map50,
                     mAP50_95=r.box.map, P=r.box.mp, R=r.box.mr, ms_GPU=r.speed['inference']))
    perkelas[nm] = {r.names[i]: r.box.maps[i] for i in r.box.ap_class_index}

log = pd.DataFrame(rows).round(3)
print('Tabel log eksperimen (val):'); display(log)
print('mAP50-95 per kelas (val):'); display(pd.DataFrame(perkelas).round(3))
log.to_csv(f'{RUNS}/log_eksperimen.csv', index=False)

## 6. Pilih model (mAP50-95 val tertinggi), uji di data TEST sekali saja

In [ ]:
terbaik = log.sort_values('mAP50_95', ascending=False).iloc[0]
nm = terbaik.eksperimen
im = {e[0]: e[2] for e in EKS}[nm]
print('Model terpilih:', nm, '| mAP50-95 val =', terbaik.mAP50_95)

t = YOLO(f'{RUNS}/{nm}/weights/best.pt').val(
    data=D, split='test', imgsz=im, plots=True, project=RUNS, name='TEST_' + nm, exist_ok=True)
hasil_test = dict(model=nm, mAP50=round(t.box.map50, 3), mAP50_95=round(t.box.map, 3),
                  P=round(t.box.mp, 3), R=round(t.box.mr, 3))
print('HASIL TEST (sekali saja):', hasil_test)
print({t.names[i]: round(float(t.box.maps[i]), 3) for i in t.box.ap_class_index})

with open(f'{RUNS}/hasil_praktik_{NIM}.txt', 'w') as f:
    f.write(f'NIM / Nama : {NIM} / {NAMA_MHS}\nModel      : {MODEL}\n\n')
    f.write(log.to_string(index=False) + f'\n\nModel terpilih: {nm}\nHasil TEST: {hasil_test}\n')
print('Tersimpan di', RUNS)

## 7. Grafik untuk laporan

In [ ]:
from IPython.display import Image, display
for nm_ in ['E1_pretrained', 'E3_tuning']:
    for f in ['results.png', 'confusion_matrix.png']:
        print(nm_, f); display(Image(f'{RUNS}/{nm_}/{f}', width=700))

In [ ]:
# contoh prediksi pada gambar test
m = YOLO(f'{RUNS}/{nm}/weights/best.pt')
m.predict(source='/content/dataset_v1/images/test', imgsz=im, conf=0.25,
          save=True, project=RUNS, name='prediksi_test', exist_ok=True, verbose=False)
for f in sorted(glob.glob(f'{RUNS}/prediksi_test/*.jpg'))[:4]:
    display(Image(f, width=500))

## 8. (Opsional) Ekspor untuk Orange Pi 3B
Hasilnya `best.onnx`. Konversi ke RKNN (NPU RK3566) dilakukan terpisah dengan RKNN Toolkit. Ukur FPS dan latensi langsung di Orange Pi.

In [ ]:
YOLO(f'{RUNS}/{nm}/weights/best.pt').export(format='onnx', imgsz=im, opset=12, simplify=True)